# 2.3 因果掩码与三类架构 / Causal masking and the three architectures

一个三角形的掩码让模型不能偷看答案，也决定了 GPT、BERT 和 T5 的区别。

A triangular mask keeps the model from peeking at the answer, and it also defines the difference between GPT, BERT and T5.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 实现因果掩码并用扰动测试验证 / Implement the causal mask and verify it by perturbation

实现下三角掩码和带掩码的 softmax。然后写一个通用的检测函数，把某个位置之后的输入换成随机值，观察它之前的输出是否变化。测试会在一个两层小模型上确认带掩码时变化为 0、不带掩码时有变化，最后在真实的 GPT-2 上验证 12 层叠加后仍然没有泄漏。

Implement the lower-triangular mask and a masked softmax. Then write a general detector that replaces the input after some position with random values and observes whether the outputs before it change. The tests confirm on a small two-layer model that the change is 0 with the mask and nonzero without it, and finally verify on the real GPT-2 that nothing leaks through 12 stacked layers.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch


def causal_mask(T):
    """Bool tensor (T, T). Entry [i, j] is True when position i may attend to position j."""
    # TODO 1: position i sees itself and everything before it
    raise NotImplementedError


def masked_softmax(scores, mask):
    """Softmax over the last dimension, with zero weight wherever mask is False."""
    # TODO 2: fill with -inf before the softmax (not zero afterwards: the rows must still sum to 1)
    raise NotImplementedError


@torch.no_grad()
def future_leak(f, x, pos):
    """Does the output before `pos` depend on the input from `pos` onwards?

    f maps a tensor (T, ...) to a tensor (T, ...), one output row per position.
    Replace x[pos:] by random values of the same shape and dtype, run f on both
    versions, and return the largest absolute change in the outputs at positions < pos.
    (For an integer x, draw random token ids with torch.randint_like(x[pos:], 50257).)
    """
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

m = causal_mask(4)
assert m.dtype == torch.bool and m.tolist() == [[True, False, False, False], [True, True, False, False],
                                                [True, True, True, False], [True, True, True, True]]

torch.manual_seed(0)
s = torch.randn(4, 4, dtype=torch.float64)
w = masked_softmax(s, m)
assert torch.allclose(w.sum(-1), torch.ones(4, dtype=torch.float64)), "rows must still sum to 1"
assert (w[~m] == 0).all(), "future positions must get exactly zero weight"
assert w[0, 0] == 1.0, "the first position can only attend to itself"
assert torch.allclose(w[3], torch.softmax(s[3], -1)), "the last row sees everything, so it is an ordinary softmax"

# A tiny two-layer attention-only model, with and without the mask.
T, d = 8, 16
W = [torch.randn(3, d, d, dtype=torch.float64) / math.sqrt(d) for _ in range(2)]
def tiny(x, masked):
    for wq, wk, wv in W:
        scores = (x @ wq) @ (x @ wk).T / math.sqrt(d)
        mask = causal_mask(T) if masked else torch.ones(T, T, dtype=torch.bool)
        x = x + masked_softmax(scores, mask) @ (x @ wv)
    return x
x = torch.randn(T, d, dtype=torch.float64)
assert future_leak(lambda t: tiny(t, True), x, pos=5) < 1e-12, "with a causal mask, earlier outputs must not change at all"
assert future_leak(lambda t: tiny(t, False), x, pos=5) > 1e-3, "without the mask, earlier outputs do depend on later inputs"
assert torch.equal(x, x.clone()) and future_leak(lambda t: t, x, pos=5) == 0.0

# The real thing: GPT-2 is causal through all 12 layers.
tok = AutoTokenizer.from_pretrained("openai-community/gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").eval()
ids = tok("The capital of France is Paris and the capital of Germany is Berlin", return_tensors="pt").input_ids[0]
leak = future_leak(lambda t: gpt2(t[None]).logits[0], ids, pos=6)
assert leak < 1e-3, f"GPT-2 logits before position 6 changed by {leak}"
print(f"m23_mask: all tests passed (GPT-2 leak {leak:.1e})")

<details><summary><b>提示 1 / Hint 1</b></summary>

`torch.tril(torch.ones(T, T, dtype=torch.bool))` 直接给出下三角的布尔矩阵。

`torch.tril(torch.ones(T, T, dtype=torch.bool))` gives the lower-triangular boolean matrix directly.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

`future_leak` 中先 `x2 = x.clone()`，再替换 `x2[pos:]`。浮点输入用 `torch.randn_like`，整数输入用 `torch.randint_like(x[pos:], 50257)`，可以用 `x.is_floating_point()` 区分。

In `future_leak` start with `x2 = x.clone()` and replace `x2[pos:]`. Use `torch.randn_like` for float input and `torch.randint_like(x[pos:], 50257)` for integer input, telling them apart with `x.is_floating_point()`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

返回 `(f(x)[:pos] - f(x2)[:pos]).abs().max().item()`。

Return `(f(x)[:pos] - f(x2)[:pos]).abs().max().item()`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


def causal_mask(T):
    return torch.tril(torch.ones(T, T, dtype=torch.bool))


def masked_softmax(scores, mask):
    return torch.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)


@torch.no_grad()
def future_leak(f, x, pos):
    x2 = x.clone()
    if x.is_floating_point():
        x2[pos:] = torch.randn_like(x[pos:])
    else:
        x2[pos:] = torch.randint_like(x[pos:], 50257)
    return (f(x)[:pos] - f(x2)[:pos]).abs().max().item()
```

</details>

## 2. 架构选择与掩码自测 / Self-check on architectures and masks

不看上面的内容回答。
(a) 下面四个任务各自最自然的架构是哪一类？情感分类，英译中，多轮聊天，为检索生成文本向量。
(b) 序列长度为 5 时，因果掩码下分数矩阵中有多少个位置是允许的？长度为 $T$ 时呢？
(c) 如果先做 softmax，再把未来位置的权重改成 0，会出什么问题？
(d) encoder-decoder 模型中，encoder 的输入有 7 个 token，decoder 已生成 4 个 token。交叉注意力的权重矩阵是什么形状？

Answer without looking at the text above.
(a) Which class of architecture is the most natural for each of these four tasks? Sentiment classification, English to Chinese translation, multi-turn chat, producing text vectors for retrieval.
(b) With a sequence of length 5, how many entries of the score matrix are allowed under the causal mask? And with length $T$?
(c) What goes wrong if softmax is applied first and the weights of future positions are set to 0 afterwards?
(d) In an encoder-decoder model the encoder input has 7 tokens and the decoder has generated 4 tokens. What is the shape of the cross-attention weight matrix?

<details><summary><b>提示 1 / Hint 1</b></summary>

(b) 数一数下三角（含对角线）有多少个元素。

(b) Count the entries of the lower triangle including the diagonal.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(d) 行对应发出 query 的位置，列对应提供 key 的位置。

(d) Rows correspond to the positions that issue queries, columns to the positions that provide keys.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 情感分类用 encoder-only。英译中用 encoder-decoder。多轮聊天用 decoder-only。检索向量传统上用 encoder-only。需要说明的是，这四个任务如今都可以由 decoder-only 的大模型完成，这里问的是结构上最直接的对应。

(b) 允许的位置有 $1 + 2 + 3 + 4 + 5 = 15$ 个，总共 25 个。长度为 $T$ 时是 $T(T+1)/2$。

(c) 每一行剩下的权重之和小于 1，而且越靠前的位置被去掉的越多，输出向量的大小会随位置变化。把掩码加在 softmax 之前，权重会在允许的位置上重新归一化。

(d) 形状是 $(4, 7)$。query 来自 decoder 的 4 个位置，key 和 value 来自 encoder 的 7 个位置。

---

(a) Sentiment classification uses encoder-only. English to Chinese translation uses encoder-decoder. Multi-turn chat uses decoder-only. Retrieval vectors traditionally use encoder-only. Note that all four tasks can now be done by decoder-only large models. The question asks for the most direct structural match.

(b) $1 + 2 + 3 + 4 + 5 = 15$ entries are allowed out of 25. With length $T$ the count is $T(T+1)/2$.

(c) The remaining weights in each row sum to less than 1, and earlier positions lose more, so the size of the output vector would vary with position. Applying the mask before softmax renormalizes the weights over the allowed positions.

(d) The shape is $(4, 7)$. The queries come from the 4 decoder positions and the keys and values from the 7 encoder positions.

</details>